# NVS Injection — Rotation Questions\n",
    "\n",
    "For each of the 200 MindCube rotation questions:\n",
    "1. Parse the rotation direction/angle from the question text\n",
    "2. Run **Zero123++** on one input image to synthesize the target rotated view\n",
    "3. Inject the synthesized view as an extra input image into LLaVA's context\n",
    "4. Evaluate whether rotation accuracy improves over baseline (34.5%)\n",
    "\n",
    "Zero123++ outputs 6 views at fixed azimuths (30°, 90°, 150°, 210°, 270°, 330°).\n",
    "We select the view closest to the parsed rotation angle.\n",
    "\n",
    "We run in two phases to avoid OOM:\n",
    "- **Phase 1:** Load Zero123++ → synthesize all 200 views → save to **Google Drive** → unload\n",
    "- **Phase 2:** Load LLaVA → run inference, reading synthesized views from Drive\n",
    "\n",
    "Everything is saved to Drive — no large files written to local Colab disk.\n",
    "\n",
    "**Runtime:** A100 GPU (40 GB). Output: `MyDrive/MindCube/nvs_results/nvs_injection_results.jsonl`

In [ ]:
# ── 1. Install ───────────────────────────────────────────────────────────────
!pip install -q diffusers transformers accelerate pillow tqdm huggingface_hub einops

In [ ]:
# ── 2. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = "/content/drive/MyDrive/MindCube/data/raw/MindCube_tinybench.jsonl"
IMAGE_ROOT = "/content/drive/MyDrive/MindCube/data/"
LLAVA_ID   = "/content/drive/MyDrive/models/llava-onevision-qwen2-7b-ov-hf"

# All outputs go to Drive — nothing large written to local Colab disk
NVS_DIR    = "/content/drive/MyDrive/MindCube/nvs_views"      # synthesized PNGs
OUT_DIR    = "/content/drive/MyDrive/MindCube/nvs_results"    # JSONL + figures
OUT_PATH   = f"{OUT_DIR}/nvs_injection_results.jsonl"
SANITY_PNG = f"{OUT_DIR}/nvs_sanity.png"

import pathlib, os
assert pathlib.Path(DATA_PATH).exists()
os.makedirs(NVS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("Drive mounted. Output dirs ready.")
print(f"  NVS views  → {NVS_DIR}")
print(f"  Results    → {OUT_PATH}")

In [ ]:
# ── 3. Rotation parser ────────────────────────────────────────────────────────
# Zero123++ fixed azimuths (relative to input view, degrees)
# Grid layout: 2 rows x 3 cols
#   Row 0 (elev +20°): col0=30°, col1=90°, col2=150°
#   Row 1 (elev -20°): col0=210°, col1=270°, col2=330°
import re
import numpy as np

_Z123_AZIMUTHS = [30, 90, 150, 210, 270, 330]  # indices 0-5
_Z123_GRID = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]  # (row, col)


def parse_rotation_degrees(question: str) -> float | None:
    """
    Extract a signed azimuth angle (degrees) from a MindCube rotation question.
    Clockwise → positive azimuth. Counter-clockwise → negative (mapped to 360-x).
    Returns None if no rotation can be parsed.
    """
    q = question.lower()

    # Extract numeric angle
    angle_match = re.search(r'(\d+)\s*(?:degree|°)', q)
    angle = int(angle_match.group(1)) if angle_match else 90  # default 90°

    # Determine direction
    cw_keywords  = ['clockwise', 'to the right', 'right side', 'turn right']
    ccw_keywords = ['counter', 'anti', 'to the left', 'left side', 'turn left']

    is_cw  = any(k in q for k in cw_keywords)
    is_ccw = any(k in q for k in ccw_keywords)

    if is_ccw and not is_cw:
        return (360 - angle) % 360  # e.g. 90° CCW → 270°
    else:
        return float(angle)  # CW or unknown → positive


def closest_z123_view(target_az: float) -> int:
    """Return the Zero123++ view index (0-5) closest to target_az."""
    diffs = [abs((az - target_az + 180) % 360 - 180) for az in _Z123_AZIMUTHS]
    return int(np.argmin(diffs))


def extract_z123_view(grid_image, view_idx: int):
    """
    Crop a single view from a Zero123++ output grid image (960×640, 2 rows × 3 cols).
    Returns a 320×320 PIL Image.
    """
    from PIL import Image
    W, H = grid_image.size  # (960, 640)
    w, h = W // 3, H // 2   # (320, 320)
    row, col = _Z123_GRID[view_idx]
    left, upper = col * w, row * h
    return grid_image.crop((left, upper, left + w, upper + h))


# Quick test
for q, expected in [
    ("You rotate 90 degrees clockwise.", 90),
    ("Turn 90 degrees counter-clockwise.", 270),
    ("Rotate 180 degrees.", 180),
]:
    az = parse_rotation_degrees(q)
    vi = closest_z123_view(az)
    print(f"  '{q[:40]}' → az={az}°, Z123 view {vi} ({_Z123_AZIMUTHS[vi]}°)")

In [ ]:
# ── 4. Load rotation questions ────────────────────────────────────────────────
import json
from pathlib import Path

rotation_samples = []
with open(DATA_PATH) as f:
    for line in f:
        rec = json.loads(line)
        if rec["id"].startswith("rotation"):
            rotation_samples.append(rec)

print(f"{len(rotation_samples)} rotation questions loaded.")
# Show a sample question to verify the parser
print("\nSample question:")
print(rotation_samples[0]["question"])
az = parse_rotation_degrees(rotation_samples[0]["question"])
print(f"Parsed azimuth: {az}° → Z123 view {closest_z123_view(az)} ({_Z123_AZIMUTHS[closest_z123_view(az)]}°)")

In [ ]:
# ── 5. PHASE 1: Load Zero123++ and synthesize all rotated views ───────────────
import torch, gc
from PIL import Image
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler
from tqdm.notebook import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

print("Loading Zero123++ ...")
z123_pipe = DiffusionPipeline.from_pretrained(
    "sudo-ai/zero123plus-v1.1",
    custom_pipeline="sudo-ai/zero123plus-pipeline",
    torch_dtype=torch.float16,
).to(device)
z123_pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    z123_pipe.scheduler.config, timestep_spacing="trailing"
)
print(f"Zero123++ loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

failed_ids = []

for rec in tqdm(rotation_samples, desc="Synthesizing views"):
    out_path = Path(NVS_DIR) / f"{rec['id']}.png"
    if out_path.exists():
        continue  # already synthesized (resume-safe)

    try:
        # Use the first input image as the reference view
        ref_img = Image.open(Path(IMAGE_ROOT) / rec["images"][0]).convert("RGB")

        # Parse rotation and pick the closest Zero123++ view
        az = parse_rotation_degrees(rec["question"])
        view_idx = closest_z123_view(az)

        # Synthesize
        with torch.inference_mode():
            grid = z123_pipe(ref_img, num_inference_steps=50).images[0]

        # Crop the target view and save
        view = extract_z123_view(grid, view_idx)
        view.save(str(out_path))
        ref_img.close()

    except Exception as e:
        tqdm.write(f"[WARN] {rec['id']}: {str(e)[:100]}")
        failed_ids.append(rec["id"])
        continue

    torch.cuda.empty_cache()

print(f"\nSynthesis done. {len(list(Path(NVS_DIR).glob('*.png')))} views saved.")
print(f"Failed: {len(failed_ids)}")

# Unload Zero123++ before loading LLaVA
del z123_pipe
gc.collect()
torch.cuda.empty_cache()
print(f"GPU after unload: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── 6. Sanity check: display a few synthesized views ─────────────────────────
import matplotlib.pyplot as plt

samples_to_show = list(Path(NVS_DIR).glob("*.png"))[:3]
fig, axes = plt.subplots(len(samples_to_show), 2, figsize=(8, 4 * len(samples_to_show)))

for i, nvs_path in enumerate(samples_to_show):
    rec_id = nvs_path.stem
    rec = next(r for r in rotation_samples if r["id"] == rec_id)
    ref_img = Image.open(Path(IMAGE_ROOT) / rec["images"][0])
    nvs_img = Image.open(nvs_path)
    axes[i][0].imshow(ref_img); axes[i][0].set_title(f"Reference: {rec_id[:30]}")
    axes[i][1].imshow(nvs_img); axes[i][1].set_title(
        f"Synthesized (az≈{_Z123_AZIMUTHS[closest_z123_view(parse_rotation_degrees(rec['question']))]}°)")
    for ax in axes[i]: ax.axis("off")

plt.tight_layout()
plt.savefig(SANITY_PNG, dpi=100)  # saved to Drive
plt.show()
print(f"Sanity figure saved to Drive: {SANITY_PNG}")
print("Check the synthesized views above before proceeding to Phase 2.")

In [ ]:
# ── 7. PHASE 2: Load LLaVA and run inference ──────────────────────────────────
import re
from collections import defaultdict
from transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor

_TAG  = re.compile(r"<answer>\s*([A-E])", re.I)
_DECL = re.compile(r"(?:the\s+answer\s+is|answer\s*:)\s*([A-E])\.?", re.I)
_LINE = re.compile(r"^\s*([A-E])[\.):]?\s*$", re.I | re.M)
_ANY  = re.compile(r"([A-E])", re.I)

def extract_answer(text):
    for pat in [_TAG, _DECL]:
        m = pat.search(text)
        if m: return m.group(1).upper()
    for pat in [_LINE, _ANY]:
        ms = pat.findall(text)
        if ms: return ms[-1].upper()
    return None

_HEADER = "Look at these images carefully. They show a scene from different viewpoints.\n\n"
_FOOTER = "\n\nAnswer with one letter only (A, B, C, or D)."

print("Loading LLaVA ...")
processor = AutoProcessor.from_pretrained(LLAVA_ID)
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    LLAVA_ID, torch_dtype=torch.float16, device_map="auto", attn_implementation="sdpa",
)
model.eval()
processor.image_processor.do_image_splitting = False
gc.collect()
torch.cuda.empty_cache()
print(f"LLaVA loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")


@torch.inference_mode()
def generate(images, prompt, max_new_tokens=128):
    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": prompt}]
    conversation = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=images, text=text, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


print("LLaVA ready.")

In [ ]:
# ── 8. Sanity check inference (3 samples) ────────────────────────────────────
for rec in rotation_samples[:3]:
    nvs_path = Path(NVS_DIR) / f"{rec['id']}.png"
    rgb_imgs = [Image.open(Path(IMAGE_ROOT) / r).convert("RGB") for r in rec["images"]]

    if nvs_path.exists():
        nvs_img = Image.open(nvs_path).convert("RGB")
        all_imgs = rgb_imgs + [nvs_img]
        n_rgb = len(rgb_imgs)
        note = (f" The last image is a synthesized view showing the scene after "
                f"the described rotation (generated by a 3D model).")
        prompt = _HEADER + rec["question"] + note + _FOOTER
    else:
        all_imgs = rgb_imgs
        prompt = _HEADER + rec["question"] + _FOOTER

    raw = generate(all_imgs, prompt)
    pred = extract_answer(raw)
    nvs_used = nvs_path.exists()
    print(f"[{rec['id']}]  gt={rec['gt_answer']}  pred={pred}  nvs={nvs_used}  raw={repr(raw[:60])}")
    for img in all_imgs:
        img.close()

In [ ]:
# ── 9. Full evaluation on all 200 rotation questions ─────────────────────────
OUT_PATH = "/content/nvs_injection_results.jsonl"
BASELINE = 0.345

results = []
with open(OUT_PATH, "w") as f_out:
    for rec in tqdm(rotation_samples, desc="Evaluating"):
        nvs_path = Path(NVS_DIR) / f"{rec['id']}.png"
        rgb_imgs = []
        try:
            rgb_imgs = [Image.open(Path(IMAGE_ROOT) / r).convert("RGB") for r in rec["images"]]
            nvs_used = nvs_path.exists()

            if nvs_used:
                nvs_img = Image.open(nvs_path).convert("RGB")
                all_imgs = rgb_imgs + [nvs_img]
                note = (" The last image is a synthesized view showing the scene "
                        "after the described rotation (generated by a 3D model).")
                prompt = _HEADER + rec["question"] + note + _FOOTER
            else:
                all_imgs = rgb_imgs
                prompt = _HEADER + rec["question"] + _FOOTER

            raw = generate(all_imgs, prompt)
            predicted = extract_answer(raw)
            error = None
        except Exception as e:
            raw, predicted, error = "", None, str(e)
            nvs_used = False
            tqdm.write(f"[WARN] {rec['id']}: {str(e)[:100]}")

        for img in rgb_imgs:
            img.close()

        gt = (rec["gt_answer"] or "").upper()
        result = {
            "id": rec["id"],
            "gt_answer": gt,
            "predicted": predicted,
            "correct": predicted is not None and predicted == gt,
            "nvs_used": nvs_used,
            "raw_output": raw,
            **(({"error": error}) if error else {}),
        }
        results.append(result)
        f_out.write(json.dumps(result) + "\n")

print(f"Done. Saved to {OUT_PATH}")

In [ ]:
# ── 10. Metrics ───────────────────────────────────────────────────────────────
with_nvs    = [r for r in results if r["nvs_used"]]
without_nvs = [r for r in results if not r["nvs_used"]]

acc = lambda rs: sum(r["correct"] for r in rs) / len(rs) if rs else 0.0

print(f"\n{'='*52}")
print(f"  NVS Injection — Rotation only (baseline = {BASELINE})")
print(f"{'='*52}")
print(f"  Samples with NVS view    : {len(with_nvs)}")
print(f"  Samples without NVS view : {len(without_nvs)}  (synthesis failed)")
print(f"")
print(f"  Baseline accuracy          : {BASELINE:.3f}  (69/200)")
overall_acc = acc(results)
print(f"  Overall accuracy (NVS+no)  : {overall_acc:.3f}  delta={overall_acc-BASELINE:+.3f}")
if with_nvs:
    nvs_acc = acc(with_nvs)
    print(f"  Accuracy (NVS only, n={len(with_nvs):3d})  : {nvs_acc:.3f}  delta={nvs_acc-BASELINE:+.3f}")
print(f"{'='*52}")

unanswered = sum(1 for r in results if r["predicted"] is None)
print(f"  Unanswered: {unanswered}/{len(results)}")

delta = overall_acc - BASELINE
if delta > 0.05:
    print("\n>> Substantial improvement: NVS supplies the missing viewpoint simulation.")
elif delta > 0.02:
    print("\n>> Modest improvement: NVS partially helps.")
else:
    print("\n>> Negligible improvement: viewpoint simulation bottleneck is compositional,")
    print("   not solvable by supplying the target view as an image.")

In [ ]:
# ── 10b. Diagnostic: does the NVS image change any individual answers? ────────
# If base and nvs outputs are identical for every sample, the model is ignoring
# the synthesized image entirely — likely a synthesis quality issue.
print("Running 5-sample diagnostic (baseline vs. NVS side-by-side)...\n")

for rec in rotation_samples[:5]:
    nvs_path = Path(NVS_DIR) / f"{rec['id']}.png"
    rgb_imgs = [Image.open(Path(IMAGE_ROOT) / r).convert("RGB") for r in rec["images"]]

    # Without NVS
    prompt_base = _HEADER + rec["question"] + _FOOTER
    raw_base = generate(rgb_imgs, prompt_base)

    # With NVS
    nvs_img = Image.open(nvs_path).convert("RGB")
    note = (" The last image is a synthesized view showing the scene after "
            "the described rotation (generated by a 3D model).")
    prompt_nvs = _HEADER + rec["question"] + note + _FOOTER
    raw_nvs = generate(rgb_imgs + [nvs_img], prompt_nvs)

    changed = "CHANGED" if raw_base != raw_nvs else "same"
    print(f"[{rec['id']}]  gt={rec['gt_answer']}  [{changed}]")
    print(f"  base: {repr(raw_base[:80])}")
    print(f"  nvs:  {repr(raw_nvs[:80])}")
    print()

    for img in rgb_imgs + [nvs_img]:
        img.close()

In [ ]:
# ── 11. Download results from Drive ──────────────────────────────────────────
# Results are already on Drive at OUT_PATH and SANITY_PNG.
# Use files.download() to also get a local copy in your browser.
from google.colab import files
files.download(OUT_PATH)
files.download(SANITY_PNG)